# Local testing — ML delivery template

Run every check for this template repo from one place: unit tests, the full suite,
formatting, the offline smoke test (both the `mlops_stacks_native` and `kedro` project
skeletons), workflow linting, and the deployment-repo scope logic.

**Prerequisites**
- Python 3.9+ with `pip`
- [Databricks CLI](https://docs.databricks.com/aws/en/dev-tools/cli/) ≥ v0.236.0 (`databricks --version`)
- Optional: [`actionlint`](https://github.com/rhysd/actionlint) and [`act`](https://github.com/nektos/act) for workflow linting / the `--large` suite

Notebook outputs are stripped on commit via `nbstripout` (see the setup cell), so committing this notebook won't leak run output.

In [ ]:
# Always run from the repo root, wherever this notebook lives.
import os, subprocess
root = subprocess.check_output(["git", "rev-parse", "--show-toplevel"]).decode().strip()
os.chdir(root)
print("working dir:", os.getcwd())

## 1. One-time setup
Installs dev dependencies and enables `nbstripout` as a git filter so notebook outputs are stripped automatically on commit.

In [ ]:
!pip install -r dev-requirements.txt
!nbstripout --install

## 2. Unit tests
Fast, no Databricks workspace needed. Expected: `44 passed, 9 skipped` (the 9 skips are the `--large` tests; the count includes `test_generate_project_skeleton_flavor`, parametrized over both project skeletons).

In [ ]:
!pytest tests -q

## 3. Offline smoke test
Renders the template locally (no workspace/auth) for **both project skeletons** —
`mlops_stacks_native` (default) and `kedro` — across both the `Project_Only` and
`CICD_and_Project` flows (4 renders total). Checks for un-rendered template leaks, parses all
generated YAML, runs `bundle validate` up to the auth boundary for each skeleton, and lists
remaining `TODO_` placeholders. Output is kept under `tmp/smoke-test/` (git-ignored) for inspection.

In [ ]:
!./scripts/smoke-test.sh cdi promo sales

In [ ]:
# Inspect what the smoke test generated — mlops_stacks_native skeleton
!find tmp/smoke-test/project_only_native -maxdepth 4 -type d
!echo '--- databricks.yml ---'
!cat tmp/smoke-test/project_only_native/domains/cdi/promo/cdi_promo/databricks.yml

## 4. The Kedro skeleton

Selecting `input_project_skeleton: kedro` scaffolds a real Kedro project
(`src/<project>/pipelines/{data_processing,model_training,model_inference}`, `conf/`,
`pyproject.toml`) instead of the native `training/`/`deployment/`/`validation/`/`monitoring/`
layout — the two are mutually exclusive (see `docs/architecture-decisions.md` ADR-17).

Both skeletons share the same DAB deploy layer (`databricks.yml`, the MLflow resource, the
GitHub Actions workflows); only the job **task type** differs — `notebook_task` for native vs
`python_wheel_task` calling `kedro run --pipeline <name> --env ${bundle.target}` for kedro.

In [ ]:
# Inspect what the smoke test generated — kedro skeleton
!find tmp/smoke-test/project_only_kedro -maxdepth 5 -type d
!echo '--- pyproject.toml ---'
!cat tmp/smoke-test/project_only_kedro/domains/cdi/promo/cdi_promo/pyproject.toml

In [ ]:
# Compare the job task type between skeletons: notebook_task (native) vs python_wheel_task (kedro)
!echo '=== native: model-workflow-resource.yml (Train task) ==='
!sed -n '/task_key: Train/,/task_key: ModelValidation/p' tmp/smoke-test/project_only_native/domains/cdi/promo/cdi_promo/resources/model-workflow-resource.yml | head -20
!echo
!echo '=== kedro: model-workflow-resource.yml (Train task) ==='
!sed -n '/task_key: Train/,/environments:/p' tmp/smoke-test/project_only_kedro/domains/cdi/promo/cdi_promo/resources/model-workflow-resource.yml | head -20

In [ ]:
%%bash
# Run the kedro project's own placeholder unit tests (needs kedro installed locally; skips cleanly if not)
cd tmp/smoke-test/project_only_kedro/domains/cdi/promo/cdi_promo
if python3 -c "import kedro" 2>/dev/null; then
  pip install -e . -q
  pytest -q
else
  echo 'kedro not installed locally — skipping (this is fine; CI installs it via pyproject.toml)'
fi

## 5. Lint the generated GitHub Actions workflows
Requires [`actionlint`](https://github.com/rhysd/actionlint) on your PATH. Lints the workflows for both skeletons, generated by the smoke test.

In [ ]:
!command -v actionlint >/dev/null && actionlint tmp/smoke-test/with_cicd_native/domains/cdi/promo/.github/workflows/*.yml || echo 'actionlint not installed — skipping'

In [ ]:
!command -v actionlint >/dev/null && actionlint tmp/smoke-test/with_cicd_kedro/domains/cdi/promo/.github/workflows/*.yml || echo 'actionlint not installed — skipping'

## 6. Deployment-repo scope logic
Confirms the scoped-deploy matrix from `deployment-repo/bundles.yml`: the `dev` branch maps `cdi_promo` to the
`dev` target, while `main` (which `cdi_promo` doesn't opt into) yields nothing.

In [ ]:
%%bash
cd deployment-repo
echo '== dev branch (expect cdi_promo/dev) =='
GITHUB_REF_NAME=dev DISPATCH_PROJECT=cdi_promo GITHUB_OUTPUT=/dev/null python3 .github/scripts/monorepo_scope.py
echo '== main branch (expect empty) =='
GITHUB_REF_NAME=main DISPATCH_PROJECT=cdi_promo GITHUB_OUTPUT=/dev/null python3 .github/scripts/monorepo_scope.py

## 7. Full suite (unit + integration)
The `--large` tests need [`act`](https://github.com/nektos/act) and a real Databricks workspace, so they are
**not** run above. Run this only in an environment set up for them (also runs `black` formatting via `--black`).

In [ ]:
# Heavy — needs act + Databricks auth. Uncomment to run.
# !pytest tests --large -vv --black

## 8. Validate a generated bundle against dev_qa (needs auth)
Authenticate the CLI once, then validate the `dev` target of a smoke-test-generated project against the
dev_qa workspace. `validate` is safe (read-only); `deploy` would create resources. Works the same for
either skeleton — just point at `project_only_native` or `project_only_kedro`.

In [ ]:
# One-time interactive login (opens a browser). Uncomment to run.
# !databricks auth login --host https://dbc-26cf5e15-d40d.cloud.databricks.com --profile dev_qa

# Then validate the generated bundle (swap _native for _kedro to check the other skeleton):
# !cd tmp/smoke-test/project_only_native/domains/cdi/promo/cdi_promo && databricks bundle validate -t dev -p dev_qa